# Collaborative filtering with k-Nearest Neighbours

Collaborative filtering is another method of recommending content to users. There are two ways of tackling this:

 - **User-based** collaborative filtering is based on the user similarity or neighborhood
 - **Item-based** collaborative filtering is based on similarity among items

 User-based collaborative filtering looks for users who are similar. This is very similar to the user clustering method done previously; where we employed explicit user profiles to calculate user similarity. However, the user profiles may not be available, so how can we determine if two users are similar?

We'll again look at this in terms of recommending courses based on user ratings. Generally we would want to predict new items to users, so we can look at items they have not rated before. Then we can predict the rating for novel items based on the similarity of the user to other users.

Let's assume we have a dataset of user rows, and course columns. Each entry is then the rating of a user for a course. When we then look at the rows and compare them to each other, we are doing **User-based** filtering, described by

$$\hat{r}_{ui} = \frac{
\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v) \cdot r_{vi}}
{\sum\limits_{v \in N^k_i(u)} \text{similarity}(u, v)}$$

if we look at the columns we are doing **Item-based** filtering, described by

$$\hat{r}_{ui} = \frac{
\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j) \cdot r_{uj}}
{\sum\limits_{j \in N^k_u(i)} \text{similarity}(i, j)}$$

Where the sum goes over $N^k_i(u)$, the k nearest neighbours of $u$.

In [ ]:
import numpy as np
import math
import pandas as pd

As an example, we have an example similarities array for users and a list of ratings. Taking the inner proudct and normalising by the total similarity score gives a prediction for the expected rating of the user for that item.

In [ ]:
# An example similarity array stores the similarity of user2, user3, user4, and user5 to user6
knn_sims = np.array([0.8, 0.92, 0.75, 0.83])

# 2.0 means audit and 3.0 means complete the course
knn_ratings = np.array([3.0, 3.0, 2.0, 3.0]) 

r_u6_ml =  np.dot(knn_sims, knn_ratings)/ sum(knn_sims)
true_rating = 3.0
rmse = math.sqrt(true_rating - r_u6_ml) ** 2
r_u6_ml, rmse

(2.7727272727272725, 0.22727272727272751)

Next, let's apply this to a real dataset.

In [ ]:
rating_df = pd.read_csv("./data/ratings.csv")
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


First, let's reformat the dataset in the expected way, with the users as the rows and the courses as the columns. The original data is representeed by a dense format, where the only entries are the actual ratings. To easily take inner products we want to get the data in a sparse format.

In [ ]:
rating_sparse_df = rating_df.pivot(
    index='user', 
    columns='item', 
    values='rating'
    ).fillna(0).reset_index().rename_axis(index=None, columns=None)
rating_sparse_df.head()

,user,AI0111EN,BC0101EN,BC0201EN,BC0202EN,BD0101EN,BD0111EN,BD0115EN,BD0121EN,BD0123EN,...,SW0201EN,TA0105,TA0105EN,TA0106EN,TMP0101EN,TMP0105EN,TMP0106,TMP107,WA0101EN,WA0103EN
0,2,0.0,4.0,0.0,0.0,5.0,4.0,0.0,5.0,3.0,...,0.0,5.0,0.0,4.0,0.0,3.0,3.0,0.0,5.0,0.0
1,4,0.0,0.0,0.0,0.0,5.0,3.0,4.0,5.0,3.0,...,0.0,4.0,0.0,0.0,0.0,3.0,3.0,0.0,3.0,3.0
2,5,3.0,5.0,5.0,0.0,4.0,0.0,0.0,0.0,3.0,...,0.0,0.0,4.0,4.0,4.0,4.0,4.0,5.0,0.0,3.0
3,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Scikit-Surprise implementation

scikit-surprise is a good library for making recommender systems that makes it very easy to build and test them.

In [ ]:
from surprise import KNNBasic
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

Using a dataset from the surprise module shows how easy it is to build a recommender system.

In [ ]:
# Load the movielens-100k dataset (download it if needed),
data = Dataset.load_builtin('ml-100k', prompt=False)

# sample random trainset and testset
# test set is made of 25% of the ratings.
trainset, testset = train_test_split(data, test_size=.25)

# We'll use the famous KNNBasic algorithm.
algo = KNNBasic()

# Train the algorithm on the trainset, and predict ratings for the testset
algo.fit(trainset)
predictions = algo.test(testset)

# Then compute RMSE
accuracy.rmse(predictions)

Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.9731


0.9730720112522221

Let;s now try with our own dataset. For this we first save the data to a new csv file, then use the Reader class from surprise to load the dataset in a format that the library can work with.

In [ ]:
# Save the rating dataframe to a CSV file
rating_df.to_csv("./data/course_ratings.csv", index=False)

# Read the course rating dataset with columns user item rating
reader = Reader(
    line_format='user item rating', sep=',', skip_lines=1, rating_scale=(2, 3)
)

# Load the dataset from the CSV file
course_dataset = Dataset.load_from_file("./data/course_ratings.csv", reader=reader)

trainset, testset = train_test_split(course_dataset, test_size=.3)
print(f"Total {trainset.n_users} users and {trainset.n_items} items in the trainingset")

Total 31310 users and 122 items in the trainingset


In [ ]:
## WRITE YOUR CODE HERE:
recommender_model = KNNBasic(sim_options={'name': 'cosine', 'user_based': False})
recommender_model.fit(trainset)
preds = recommender_model.test(testset)
accuracy.rmse(preds)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.2865


1.286491577256459